# CW2 Scaffold A: Blockchain Fraud Detection

Rare-event classification on the Elliptic Bitcoin dataset.

**Module:** FIN306 / FIN510

**Instructions:** Run provided cells, complete `# TODO` sections, write a 2,500-word reflective report.

**Data:** Elliptic Bitcoin labelled subset — loaded automatically from `fin510-colab-notebooks/resources/elliptic/` on Colab, or from `data/elliptic/` if you have cloned the course repo locally. No manual download required.

## Before you code: state your research claim

Good data science starts with a question that evidence can refute. Before running any cells, read the claim below and ask yourself the four questions that follow.

---

**Starting claim:**

> *Standard shuffled cross-validation overstates fraud detection AUC on the Elliptic Bitcoin dataset relative to temporal walk-forward validation, because fraud typologies evolve over time and shuffled CV allows future patterns to leak into training.*

---

**Think through these questions before you start:**

1. **Why would you expect this to be true?** What is it about fraud — as opposed to, say, credit scoring on cross-sectional data — that makes temporal ordering especially important?

2. **What would falsify it?** If shuffled AUC and walk-forward AUC are nearly identical on this dataset, what would that tell you about the nature of temporal drift in the Elliptic data?

3. **Is statistical significance enough?** Suppose you find a gap of 0.065 AUC. Is that economically meaningful? Think about what a 6.5 percentage-point difference in AUC means when a bank screens 50,000 transactions per day.

4. **What alternative explanations exist?** Could the gap be driven by class imbalance changing over time, rather than fraud pattern evolution? How would you distinguish between these?

---

Write a one- or two-sentence version of *your own* refined claim in the markdown cell below — before running any code. Your report introduction should open with this claim.

**Your refined claim (write here before running any code):**

*Replace this text with your own one- or two-sentence falsifiable claim. You will return to this cell after completing the analysis and assess whether your results support or refute it.*

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve,
)
from pathlib import Path

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


## Step 1: Load Elliptic (labelled subset)

In [ ]:
ELL_URL = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "main/resources/elliptic/elliptic_labelled_v2.parquet"
)
# Try common local paths (repo clone, lab working dir) before falling back to URL.
ell_path = None
for _p in [
    Path("data/elliptic/elliptic_labelled.parquet"),
    Path("../data/elliptic/elliptic_labelled.parquet"),
    Path("resources/elliptic/elliptic_labelled.parquet"),
    Path("elliptic_labelled.parquet"),
]:
    if _p.is_file():
        ell_path = _p
        break
if ell_path is None:
    ell_path = Path("elliptic_labelled.parquet")
    import urllib.request
    print("Downloading Elliptic labelled parquet from GitHub (~25 MB)...")
    urllib.request.urlretrieve(ELL_URL, ell_path)
    print("Done.")

print(f"Loaded from: {ell_path}")
ell = pd.read_parquet(ell_path)
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values

print(f"Transactions:  {len(ell):,}")
print(f"Features:      {len(feat_cols)}")
print(f"Illicit:       {y.sum():,} ({y.mean():.1%})")
print(f"Time steps:    {ts.min()} to {ts.max()}")


### TODO 1: Plot illicit rate by `time_step` (temporal drift).

In your report, explain why drift matters for evaluation.

In [ ]:
# TODO 1: your plot here


## Step 2: Baseline (shuffled stratified CV)

In [ ]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_shuffled = cross_val_score(pipe_lr, X, y, cv=cv, scoring="roc_auc")
ap_shuffled = cross_val_score(pipe_lr, X, y, cv=cv, scoring="average_precision")
print("Shuffled 5-fold CV (logistic, balanced)")
print(f"  AUC: {auc_shuffled.mean():.3f} +/- {auc_shuffled.std():.3f}")
print(f"  AP:  {ap_shuffled.mean():.3f} +/- {ap_shuffled.std():.3f}")


## Step 3: Walk-forward temporal validation

In [ ]:
boundaries = [1, 10, 20, 30, 40, 50]
wf_results = []
for i in range(len(boundaries) - 2):
    train_mask = (ts >= boundaries[0]) & (ts < boundaries[i + 1])
    test_mask = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
    if train_mask.sum() < 50 or test_mask.sum() < 50:
        continue
    X_tr, y_tr = X[train_mask], y[train_mask]
    X_te, y_te = X[test_mask], y[test_mask]
    pipe_lr.fit(X_tr, y_tr)
    auc_wf = roc_auc_score(y_te, pipe_lr.predict_proba(X_te)[:, 1])
    wf_results.append({
        "window": f"t={boundaries[i+1]}-{boundaries[i+2]-1}",
        "AUC": auc_wf,
        "illicit_rate": y_te.mean(),
    })
    print(f"  {wf_results[-1]['window']}: AUC={auc_wf:.3f} illicit={y_te.mean():.1%}")

mean_wf = np.mean([r["AUC"] for r in wf_results])
print(f"\nWalk-forward AUC (mean): {mean_wf:.3f}")
print(f"Shuffled CV AUC:         {auc_shuffled.mean():.3f}")
print(f"Look-ahead bias gap:     {auc_shuffled.mean() - mean_wf:+.3f}")


### TODO 2: Dual-axis plot of walk-forward AUC and illicit rate per window.

Add horizontal line for shuffled CV AUC.

In [ ]:
# TODO 2: your figure here


## Step 4: Cost-sensitive threshold (last WF test window)

In [ ]:
last_train = (ts >= 1) & (ts < 40)
last_test = (ts >= 40) & (ts < 50)
pipe_lr.fit(X[last_train], y[last_train])
y_prob = pipe_lr.predict_proba(X[last_test])[:, 1]
y_te = y[last_test]

cost_fp, cost_fn = 50, 5000
thresholds = np.arange(0.01, 0.95, 0.01)
costs = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel()
    costs.append((fp * cost_fp + fn * cost_fn) / len(y_te))
opt_t = thresholds[int(np.argmin(costs))]
print(f"Optimal threshold (base costs): {opt_t:.2f}")


### TODO 3: Repeat cost curves for cost_fn in {1000, 50000}; plot all three.

Discuss regulatory implications in your report.

In [ ]:
# TODO 3: your plots here


## Step 5: Model comparison (walk-forward)

In [ ]:
models = {
    "Logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight="balanced",
            random_state=42, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("clf", GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)),
    ]),
}
for name, pipe in models.items():
    aucs = []
    for i in range(len(boundaries) - 2):
        tr = (ts >= 1) & (ts < boundaries[i + 1])
        te = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
        if tr.sum() < 50 or te.sum() < 50:
            continue
        pipe.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], pipe.predict_proba(X[te])[:, 1]))
    print(f"{name:18s} mean WF AUC = {np.mean(aucs):.3f}")


### TODO 4: Grouped bar chart by model and window; interpret bias-variance and deployment risk.

In [ ]:
# TODO 4: your chart here


## Step 6 (extension): graph degrees + re-fit

In [ ]:
import networkx as nx

EDGES_URL = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "main/resources/elliptic/elliptic_edges_labelled_v2.parquet"
)
edges_path = None
for _p in [
    Path("data/elliptic/elliptic_edges_labelled.parquet"),
    Path("../data/elliptic/elliptic_edges_labelled.parquet"),
    Path("resources/elliptic/elliptic_edges_labelled.parquet"),
    Path("elliptic_edges_labelled.parquet"),
]:
    if _p.is_file():
        edges_path = _p
        break
if edges_path is None:
    edges_path = Path("elliptic_edges_labelled.parquet")
    import urllib.request
    print("Downloading Elliptic edges parquet from GitHub...")
    urllib.request.urlretrieve(EDGES_URL, edges_path)
edges_df = pd.read_parquet(edges_path)
G = nx.from_pandas_edgelist(
    edges_df, source="txId1", target="txId2", create_using=nx.DiGraph
)
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
ell = ell.copy()
ell["in_degree"] = ell["txId"].map(in_deg).fillna(0).astype(int)
ell["out_degree"] = ell["txId"].map(out_deg).fillna(0).astype(int)
ell["total_degree"] = ell["in_degree"] + ell["out_degree"]
print("Graph features merged (extension).")


### TODO 5: Stack degree features onto X; re-run walk-forward for your best model.

In [ ]:
# TODO 5: your extension here
